In [ ]:
import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
### cell 0 ###

date = pd.Timestamp("1998-09-02")
lineitem_filtered = lineitem.loc[
    :,
    [
        "L_QUANTITY",
        "L_EXTENDEDPRICE",
        "L_DISCOUNT",
        "L_TAX",
        "L_RETURNFLAG",
        "L_LINESTATUS",
        "L_SHIPDATE",
        "L_ORDERKEY",
    ],
]
sel = lineitem_filtered.L_SHIPDATE <= date
lineitem_filtered = lineitem_filtered[sel]
lineitem_filtered["AVG_QTY"] = lineitem_filtered.L_QUANTITY
lineitem_filtered["AVG_PRICE"] = lineitem_filtered.L_EXTENDEDPRICE
lineitem_filtered["DISC_PRICE"] = lineitem_filtered.L_EXTENDEDPRICE * (
    1 - lineitem_filtered.L_DISCOUNT
)
lineitem_filtered["CHARGE"] = (
    lineitem_filtered.L_EXTENDEDPRICE
    * (1 - lineitem_filtered.L_DISCOUNT)
    * (1 + lineitem_filtered.L_TAX)
)
gb = lineitem_filtered.groupby(["L_RETURNFLAG", "L_LINESTATUS"], as_index=False)[
    [
        "L_QUANTITY",
        "L_EXTENDEDPRICE",
        "DISC_PRICE",
        "CHARGE",
        "AVG_QTY",
        "AVG_PRICE",
        "L_DISCOUNT",
        "L_ORDERKEY",
    ]
]
total = gb.agg(
    {
        "L_QUANTITY": "sum",
        "L_EXTENDEDPRICE": "sum",
        "DISC_PRICE": "sum",
        "CHARGE": "sum",
        "AVG_QTY": "mean",
        "AVG_PRICE": "mean",
        "L_DISCOUNT": "mean",
        "L_ORDERKEY": "count",
    }
)